In [1]:
print("Hello")

Hello


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Neur — Sequential Colab Pipeline

Run each cell in order. Every step reads from and writes to Google Drive so you can stop and resume between steps.

- **Step 0** — clone repo, install deps, mount Drive.
- **Step 1** — download Samanantar (2M pairs) + FLORES-200 to `Drive/neur/raw_data/`.
- **Step 2** — clean and split the raw data into `Drive/neur/processed_data/`.
- **Step 3** — train the sinusoidal model and evaluate on FLORES-200. Outputs in `Drive/neur/outputs/`.


## Step 0 — Setup (clone, install, mount Drive)

In [4]:
!git clone https://github.com/thenileshmishra/AS-RoPE.git /content/neur
%cd /content/neur

Cloning into '/content/neur'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (259/259), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 259 (delta 123), reused 208 (delta 72), pack-reused 0 (from 0)
Receiving objects: 100% (259/259), 1.06 MiB | 16.63 MiB/s, done.
Resolving deltas: 100% (123/123), done.
/content/neur


In [5]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 14.7 MB/s eta 0:00:00


In [6]:
!ls

docs  notebooks  pipeline  README.md  requirements.txt	src


In [7]:
import os, sys
os.environ['NEUR_DRIVE_ROOT'] = '/content/drive/MyDrive/neur'
sys.path.insert(0, '/content/neur')

from pipeline import paths
paths.ensure_dirs()
print(paths.summary())

PROJECT_ROOT    = /content/drive/MyDrive/neur
RAW_SAMANANTAR  = /content/drive/MyDrive/neur/raw_data/samanantar/samanantar_hi_en.tsv
RAW_FLORES      = /content/drive/MyDrive/neur/raw_data/flores200/flores200_hi_en_devtest.tsv
PROCESSED_DIR   = /content/drive/MyDrive/neur/processed_data
CHECKPOINT_DIR  = /content/drive/MyDrive/neur/outputs/checkpoints
LOGS_DIR        = /content/drive/MyDrive/neur/outputs/logs
METRICS_DIR     = /content/drive/MyDrive/neur/outputs/metrics


## Step 1 — Download raw datasets to Google Drive

Downloads up to 2M Samanantar pairs and the full FLORES-200 devtest split.
Safe to re-run — existing files are skipped unless `--force` is passed.

In [8]:
!python -m pipeline.step1_download --sample-size 2000000

[step1] paths:
PROJECT_ROOT    = /content/drive/MyDrive/neur
RAW_SAMANANTAR  = /content/drive/MyDrive/neur/raw_data/samanantar/samanantar_hi_en.tsv
RAW_FLORES      = /content/drive/MyDrive/neur/raw_data/flores200/flores200_hi_en_devtest.tsv
PROCESSED_DIR   = /content/drive/MyDrive/neur/processed_data
CHECKPOINT_DIR  = /content/drive/MyDrive/neur/outputs/checkpoints
LOGS_DIR        = /content/drive/MyDrive/neur/outputs/logs
METRICS_DIR     = /content/drive/MyDrive/neur/outputs/metrics
[step1] loading ai4bharat/samanantar/hi ...
README.md: 11.4kB [00:00, 30.1MB/s]
hi/train-00000-of-00008.parquet: 100% 240M/240M [00:03<00:00, 79.2MB/s] 
hi/train-00001-of-00008.parquet: 100% 240M/240M [00:01<00:00, 158MB/s]  
hi/train-00002-of-00008.parquet: 100% 240M/240M [00:02<00:00, 116MB/s]  
hi/train-00003-of-00008.parquet: 100% 240M/240M [00:01<00:00, 143MB/s]  
hi/train-00004-of-00008.parquet: 100% 240M/240M [00:01<00:00, 139MB/s]  
hi/train-00005-of-00008.parquet: 100% 239M/239M [00:01<00:00, 175M

## Step 2 — Clean and split the raw data

Reads `raw_data/samanantar/samanantar_hi_en.tsv`, applies cleaning filters
(length, length-ratio, script, dedupe), and writes deterministic train/val/test
splits to `processed_data/`.

In [ ]:
!python -m pipeline.step2_preprocess

## Step 3 — Train (sinusoidal) + evaluate on FLORES-200

Trains the GPT model with sinusoidal positional encoding, saving
checkpoints under `outputs/checkpoints/`, training logs under
`outputs/logs/`, and BLEU/chrF under `outputs/metrics/`.

In [ ]:
!python -m pipeline.step3_train \
  --num-steps 12000 \
  --batch-size 16 \
  --eval-every 1000 \
  --d-model 256 --n-layers 6 --n-heads 8 \
  --max-seq-len 128

## Verify outputs on Drive

In [ ]:
!ls -lh /content/drive/MyDrive/neur/outputs/checkpoints
!cat /content/drive/MyDrive/neur/outputs/metrics/metrics.json